## Imports and main

In [ ]:
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import csv
import re
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
SOURCE_DIR = Path(r"C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR, START_MONTH = 2024, 1
PROPERTY_FILTER = "Residential"
PERCENTILES = [10,25,50,75,90]
COLUMN_INDEX = {
    "CRMLSListing": 10,
    "CRMLSSold": 17
}

# Folder containing all monthly CSV files
DATA_FOLDER = Path(r"C:\Users\Viv\Documents\Career readiness\internships\IDX exchange\csv")

# Last completed calendar month
today = date.today()
last_month = today.month - 1
year = today.year
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

## Wk1: Properties filtered to residental only with row counts before & after

In [29]:


if last_month == 0:
    last_month = 12
    year -= 1


def valid_file(filename):
    stem = filename.stem

    # Match filenames like CRMLSSold202606 or CRMLSListing202401
    match = re.search(r"(\d{4})(\d{2})$", stem)

    if not match:
        return False

    file_year = int(match.group(1))
    file_month = int(match.group(2))

    if file_year < START_YEAR:
        return False

    if file_year > year:
        return False

    if file_year == year and file_month > last_month:
        return False

    return True


def combine_files(prefix):

    property_column = COLUMN_INDEX[prefix]

    combined = []
    header = None

    files = sorted(DATA_FOLDER.glob(f"{prefix}*.csv"))

    print(f"\n{prefix}")
    print("----------------------------")

    total_rows_before = 0

    for file in files:

        if not valid_file(file):
            continue

        with file.open("r", newline="", encoding="utf-8") as f:

            reader = csv.reader(f)

            file_header = next(reader)

            if header is None:
                header = file_header

            rows = list(reader)

            print(f"{file.name}: {len(rows)} rows")

            total_rows_before += len(rows)

            combined.extend(rows)

    # Row count after concatenation
    print(f"\nRows before concatenation: {total_rows_before}")
    print(f"Rows after concatenation:  {len(combined)}")

    before_filter = len(combined)

    residential = [
        row
        for row in combined
        if len(row) > property_column
        and row[property_column].strip() == "Residential"
    ]

    after_filter = len(residential)

    print(f"Rows before Residential filter: {before_filter}")
    print(f"Rows after Residential filter:  {after_filter}")

    output = DATA_FOLDER / f"{prefix}_Combined_Residential.csv"

    with output.open("w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow(header)

        writer.writerows(residential)

    print(f"Saved: {output.name}")



if __name__ == "__main__":
    combine_files("CRMLSListing")
    combine_files("CRMLSSold")


CRMLSListing
----------------------------
CRMLSListing202401.csv: 27454 rows
CRMLSListing202402.csv: 27447 rows
CRMLSListing202403.csv: 32282 rows
CRMLSListing202404.csv: 36503 rows
CRMLSListing202405.csv: 38796 rows
CRMLSListing202406.csv: 35893 rows
CRMLSListing202407.csv: 36340 rows
CRMLSListing202408.csv: 35305 rows
CRMLSListing202409.csv: 34625 rows
CRMLSListing202410.csv: 34730 rows
CRMLSListing202411.csv: 25128 rows
CRMLSListing202412.csv: 19417 rows
CRMLSListing202501.csv: 37469 rows
CRMLSListing202502.csv: 33983 rows
CRMLSListing202503.csv: 38492 rows
CRMLSListing202504.csv: 40187 rows
CRMLSListing202505.csv: 40271 rows
CRMLSListing202506.csv: 26399 rows
CRMLSListing202507.csv: 27345 rows
CRMLSListing202508.csv: 25210 rows
CRMLSListing202509.csv: 26923 rows
CRMLSListing202510.csv: 27586 rows
CRMLSListing202511.csv: 20677 rows
CRMLSListing202512.csv: 18773 rows
CRMLSListing202601.csv: 35302 rows
CRMLSListing202602.csv: 32884 rows
CRMLSListing202603.csv: 39153 rows
CRMLSListing

# WK2-3: Data from FRED merged onto sold and listings datasets using a year_month key and null count

In [ ]:

def most_recent_completed_month():
    today = date.today()
    y, m = today.year, today.month-1
    return (y-1,12) if m==0 else (y,m)

def monthly_files(prefix):
    ey, em = most_recent_completed_month()
    return [f for p in pd.period_range(f"{START_YEAR}-{START_MONTH:02d}",f"{ey}-{em:02d}",freq="M")
            if (f:=SOURCE_DIR/f"{prefix}{p.year:04d}{p.month:02d}.csv").exists()]

def process(prefix,date_col):

    files=monthly_files(prefix)
    if not files:
        print(f"No files for {prefix}")
        return None
    frames=[pd.read_csv(f,low_memory=False) for f in files]

    print("Rows before concatenation:",sum(len(x) for x in frames))

    df=pd.concat(frames,ignore_index=True)

    print("Rows after concatenation:",len(df))

    print("Unique PropertyTypes:",sorted(df.PropertyType.dropna().astype(str).str.strip().unique()))

    print("Rows before Residential filter:",len(df))


    df=df.query("PropertyType == @PROPERTY_FILTER").copy()

    print("Rows after Residential filter:",len(df))

    nulls=pd.DataFrame({"null_count":df.isna().sum()})

    nulls["null_percent"]=nulls.null_count/len(df)*100


    print(nulls)

    print("Columns >90% null:")

    print(nulls[nulls.null_percent>90])


    stats = (
    df[["ClosePrice", "LivingArea", "DaysOnMarket"]]
    .apply(pd.to_numeric, errors="coerce")
    .describe(percentiles=[p / 100 for p in PERCENTILES])
)
    print(stats)
    
    df.to_csv(SOURCE_DIR/f"{prefix}_filtered_residential.csv",index=False)

    nulls.to_csv(SOURCE_DIR/f"{prefix}_null_summary.csv")

    stats.to_csv(SOURCE_DIR/f"{prefix}_numeric_summary.csv")

    rates=pd.read_csv(URL,parse_dates=["observation_date"])

    rates.columns=["date","rate_30yr_fixed"]

    rates["year_month"]=rates.date.dt.to_period("M")

    rates=rates.groupby("year_month",as_index=False).rate_30yr_fixed.mean()

    df["year_month"]=pd.to_datetime(df[date_col],errors="coerce").dt.to_period("M")

    merged=df.merge(rates,on="year_month",how="left")

    if merged.rate_30yr_fixed.isna().any():

        raise ValueError("Mortgage-rate merge produced null values.")
    
    merged.to_csv(SOURCE_DIR/f"{prefix}_with_rates.csv",index=False)

    return merged

if __name__=="__main__":
    process("CRMLSSold","CloseDate")
    process("CRMLSListing","ListingContractDate")


Rows before concatenation: 545481
Rows after concatenation: 545481
Unique PropertyTypes: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']
Rows before Residential filter: 545481
Rows after Residential filter: 367741
                              null_count  null_percent
BuyerAgentAOR                      52791     14.355484
ListAgentAOR                       46187     12.559655
Flooring                          132234     35.958460
ViewYN                             31559      8.581855
WaterfrontYN                      367506     99.936096
...                                  ...           ...
MiddleOrJuniorSchoolDistrict      367741    100.000000
OriginatingSystemName             294467     80.074563
OriginatingSystemSubName          294467     80.074563
BuyerAgencyCompensationType       321605     87.454214
BuyerAgencyCompensation           321616     87.457205

[82 rows x 2 columns]
Colu

# WK4-5: Data Cleaning

In [ ]:
mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']


# Reporting rules:
# 1. Build a combined dataset from all monthly files in the date range.
# 2. Report unique PropertyType values found before filtering.
# 3. Apply the Residential-only filter.
# 4. Print and save null-count, missing-value, and numeric distribution summaries.
# 5. Save the filtered dataset as a new CSV.



def most_recent_completed_month():
    today = date.today()
    y, m = today.year, today.month-1
    return (y-1,12) if m==0 else (y,m)

def monthly_files(prefix):
    ey, em = most_recent_completed_month()
    return [f for p in pd.period_range(f"{START_YEAR}-{START_MONTH:02d}",f"{ey}-{em:02d}",freq="M")
            if (f:=SOURCE_DIR/f"{prefix}{p.year:04d}{p.month:02d}.csv").exists()]

def process(prefix,date_col):

    files=monthly_files(prefix)
    if not files:
        print(f"No files for {prefix}")
        return None
    frames=[pd.read_csv(f,low_memory=False) for f in files]

    print("Rows before concatenation:",sum(len(x) for x in frames))

    df=pd.concat(frames,ignore_index=True)

    print("Rows after concatenation:",len(df))

    print("Unique PropertyTypes:",sorted(df.PropertyType.dropna().astype(str).str.strip().unique()))

    print("Rows before Residential filter:",len(df))


    df=df.query("PropertyType == @PROPERTY_FILTER").copy()

    print("Rows after Residential filter:",len(df))

    nulls=pd.DataFrame({"null_count":df.isna().sum()})

    nulls["null_percent"]=nulls.null_count/len(df)*100


    print(nulls)

    print("Columns >90% null:")

    print(nulls[nulls.null_percent>90])


    stats = (
    df[["ClosePrice", "LivingArea", "DaysOnMarket"]]
    .apply(pd.to_numeric, errors="coerce")
    .describe(percentiles=[p / 100 for p in PERCENTILES])
)
    print(stats)
    
    df.to_csv(SOURCE_DIR/f"{prefix}_filtered_residential.csv",index=False)

    nulls.to_csv(SOURCE_DIR/f"{prefix}_null_summary.csv")

    stats.to_csv(SOURCE_DIR/f"{prefix}_numeric_summary.csv")

    rates=pd.read_csv(URL,parse_dates=["observation_date"])

    rates.columns=["date","rate_30yr_fixed"]

    rates["year_month"]=rates.date.dt.to_period("M")

    rates=rates.groupby("year_month",as_index=False).rate_30yr_fixed.mean()

    df["year_month"]=pd.to_datetime(df[date_col],errors="coerce").dt.to_period("M")

    merged=df.merge(rates,on="year_month",how="left")

    if merged.rate_30yr_fixed.isna().any():

        raise ValueError("Mortgage-rate merge produced null values.")
    
    merged.to_csv(SOURCE_DIR/f"{prefix}_with_rates.csv",index=False)

    return merged

if __name__=="__main__":
    process("CRMLSSold","CloseDate")
    process("CRMLSListing","ListingContractDate")

#Submit a cleaned, analysis-ready dataset as a CSV, plus a .py script documenting every transformation
# made and why. Include before/after row counts, data type confirmations, date consistency flag counts, and
# a geographic data quality summary noting any invalid coordinate records.

def most_recent_completed_month(today=None):
    today = today or date.today()
    first_day_of_current_month = date(today.year, today.month, 1)
    last_completed_month = first_day_of_current_month - pd.offsets.MonthBegin(1)
    return last_completed_month.year, last_completed_month.month


def month_range(start_year, start_month, end_year, end_month):
    start = pd.Period(f"{start_year:04d}-{start_month:02d}", freq="M")
    end = pd.Period(f"{end_year:04d}-{end_month:02d}", freq="M")
    for period in pd.period_range(start, end, freq="M"):
        yield period.year, period.month


def build_monthly_file_list(prefix, start_year, start_month, end_year, end_month):
    monthly_files = []
    for year, month in month_range(start_year, start_month, end_year, end_month):
        file_name = f"{prefix}{year:04d}{month:02d}.csv"
        file_path = SOURCE_DIR / file_name
        if file_path.exists():
            monthly_files.append(file_path)
    return monthly_files


def load_monthly_data(files):
    frames = []
    row_count_before_concat = 0

    for file_path in files:
        frame = pd.read_csv(file_path, low_memory=False)
        frames.append(frame)
        row_count_before_concat += len(frame)

    print(f"Row count before concatenation: {row_count_before_concat}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        combined = pd.DataFrame()

    print(f"Row count after concatenation: {len(combined)}")
    return combined


def get_property_column(frame):
    if "PropertyType" not in frame.columns:
        raise KeyError("PropertyType column was not found in the combined dataset.")
    return "PropertyType"

def BedAndBathValidate(frame, Bedrooms_column, Bathrooms_column): #Check for implausible values for Bedrooms and Bathrooms
    Flag = True
    try:
        if (frame[Bedrooms_column].dropna().astype(float).between(0, 20) & # implausible values for Bedrooms
            frame[Bathrooms_column].dropna().astype(float).between(0, 20) | 
            frame[Bedrooms_column].dropna().astype(float).isnull() & #Bedrooms or Bathrooms is null)
            frame[Bathrooms_column].dropna().astype(float).isnull() |  
            frame[Bathrooms_column].dropna().astype(float).greater(0) | #Bathrooms should be greater than 0
            frame[Bedrooms_column].dropna().astype(float).equals(0) & #sentinel null values
            frame[Bathrooms_column].dropna().astype(float).equals(0)): 
            Flag = False
    except Exception as e:
        print(f"Error during Bed and Bath validation: {e}")
    return Flag


def dateValidate(frame,CloseDate_column, ListingContractDate_column, PurchaseDate_column): #Date consistancy check for CloseDate and ListingContractDate
    Flag = False

    
    try:
        if (frame[CloseDate_column].dropna().astype(str).str.strip() >frame[ListingContractDate_column].dropna().astype(str).str.strip() & 
            frame[PurchaseDate_column].dropna().astype(str).str.strip() < frame[CloseDate_column].dropna().astype(str).str.strip()):
            Flag = True
    except Exception as e:
        print(f"Error during date validation: {e}")
    return Flag


def GeographicValidate(frame, lat_column, long_column): #Geographic consistancy check for Latitude and Longitude

    Flag = True
    try:
        if (frame[lat_column].dropna().astype(float).between(-90, 90) & # implausible coordinates
            frame[long_column].dropna().astype(float).between(-180, 180) | 
            frame[lat_column].dropna().astype(float).isnull() & #Latitude or Longitude is null)
            frame[long_column].dropna().astype(float).isnull() |  
            frame[long_column].dropna().astype(float).greater(0) | #California coordinates should be negative
            frame[lat_column].dropna().astype(float).equals(0) & #sentinel null values
            frame[long_column].dropna().astype(float).equals(0)): 
            Flag = False
    except Exception as e:
        print(f"Error during geographic validation: {e}")
    return Flag

def unique_property_types(frame, property_column):
    values = frame[property_column].dropna().astype(str).str.strip()
    values = values[values != ""]
    return sorted(values.unique().tolist())


def convertFieldsDateTime(frame, date_columns):
    for col in date_columns:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


def filter_residential_only(frame, property_column):
    print(f"Filtering logic applied: keep rows where {property_column} equals {PROPERTY_FILTER}.")
    before_filter = len(frame)
    filtered = frame[frame[property_column].astype(str).str.strip() == PROPERTY_FILTER].copy()
    print(f"Row count before Residential filter: {before_filter}")
    print(f"Row count after Residential filter: {len(filtered)}")
    return filtered


def build_null_summary(frame):
    summary = pd.DataFrame({
        "column": frame.columns,
        "null_count": frame.isna().sum().values,
        "row_count": len(frame),
    })
    summary["null_percent"] = np.where(
        summary["row_count"] > 0,
        (summary["null_count"] / summary["row_count"]) * 100,
        0.0,
    )
    return summary[["column", "null_count", "null_percent"]]


def build_missing_value_report(null_summary):
    flagged = null_summary[null_summary["null_percent"] > 90].copy()
    flagged.insert(0, "flag", "above_90_percent_null")
    return flagged[["flag", "column", "null_count", "null_percent"]]


def build_numeric_distribution_summary(frame, fields):
    rows = []
    for field in fields:
        if field not in frame.columns:
            rows.extend([
                {"field": field, "metric": "min", "value": np.nan},
                {"field": field, "metric": "max", "value": np.nan},
                {"field": field, "metric": "mean", "value": np.nan},
                {"field": field, "metric": "median", "value": np.nan},
                {"field": field, "metric": "p10", "value": np.nan},
                {"field": field, "metric": "p25", "value": np.nan},
                {"field": field, "metric": "p50", "value": np.nan},
                {"field": field, "metric": "p75", "value": np.nan},
                {"field": field, "metric": "p90", "value": np.nan},
            ])
            continue

        values = pd.to_numeric(frame[field], errors="coerce").dropna()
        if values.empty:
            stats = {
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "median": np.nan,
                "p10": np.nan,
                "p25": np.nan,
                "p50": np.nan,
                "p75": np.nan,
                "p90": np.nan,
            }
        else:
            percentiles = np.percentile(values, PERCENTILES)
            stats = {
                "min": values.min(),
                "max": values.max(),
                "mean": values.mean(),
                "median": values.median(),
                "p10": percentiles[0],
                "p25": percentiles[1],
                "p50": percentiles[2],
                "p75": percentiles[3],
                "p90": percentiles[4],
            }

        for metric, value in stats.items():
            rows.append({"field": field, "metric": metric, "value": value})

    return pd.DataFrame(rows)


def build_report(file_name, frame, filtered_frame):
    property_column = get_property_column(frame)
    unique_types = unique_property_types(frame, property_column)
    null_summary = build_null_summary(frame)
    missing_value_report = build_missing_value_report(null_summary)
    numeric_summary = build_numeric_distribution_summary(
        frame,
        ["ClosePrice", "LivingArea", "DaysOnMarket"],
    )

    print(f"Unique property types found for {file_name}: {unique_types}")

    report_rows = []
    report_rows.append({
        "section": "metadata",
        "item": "file_name",
        "metric": "value",
        "value": file_name,
    })
    report_rows.append({
        "section": "metadata",
        "item": "source_rows",
        "metric": "count",
        "value": len(frame),
    })
    report_rows.append({
        "section": "metadata",
        "item": "filtered_rows",
        "metric": "count",
        "value": len(filtered_frame),
    })
    report_rows.append({
        "section": "filtering_logic",
        "item": property_column,
        "metric": "rule",
        "value": f'keep rows where value equals "{PROPERTY_FILTER}"',
    })
    report_rows.append({
        "section": "unique_property_types",
        "item": property_column,
        "metric": "unique_values",
        "value": ", ".join(unique_types),
    })

    for _, row in null_summary.iterrows():
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_count",
            "value": row["null_count"],
        })
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_percent",
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in missing_value_report.iterrows():
        report_rows.append({
            "section": "missing_value_report",
            "item": row["column"],
            "metric": row["flag"],
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in numeric_summary.iterrows():
        report_rows.append({
            "section": "numeric_distribution",
            "item": row["field"],
            "metric": row["metric"],
            "value": row["value"],
        })

    report_frame = pd.DataFrame(report_rows)
    print("\nNull-count summary table:")
    print(null_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nMissing value report (columns above 90% null):")
    print(missing_value_report.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nNumeric distribution summary:")
    print(numeric_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    return report_frame


def save_outputs(file_path, file_name, filtered_frame, report_frame):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    filtered_output = file_path / f"{file_name}_filtered_residential.csv"
    report_output = file_path / f"{file_name}_calculated_summary.csv"

    filtered_frame.to_csv(filtered_output, index=False)
    report_frame.to_csv(report_output, index=False)

    print(f"\nFiltered dataset saved to {filtered_output}")
    print(f"Summary report saved to {report_output}")


def process_prefix(prefix):
    end_year, end_month = most_recent_completed_month()
    monthly_files = build_monthly_file_list(prefix, START_YEAR, START_MONTH, end_year, end_month)

    if not monthly_files:
        print(f"No monthly files found for {prefix}.")
        return

    print(
        f"Found {len(monthly_files)} monthly files for {prefix} "
        f"from {START_YEAR:04d}-{START_MONTH:02d} through {end_year:04d}-{end_month:02d}."
    )

    combined = load_monthly_data(monthly_files)
    if combined.empty:
        print(f"Combined dataset for {prefix} is empty.")
        return

    filtered = filter_residential_only(combined, get_property_column(combined))
    report_frame = build_report(prefix, combined, filtered)
    save_outputs(SOURCE_DIR, prefix, filtered, report_frame)
    return filtered

# Step 4 – Merge mortgage rates onto the combined MLS datasets
def fetch_mortgage_monthly_rates(url=URL):
    mortgage = pd.read_csv(url, parse_dates=["observation_date"])
    mortgage.columns = ["date", "rate_30yr_fixed"]
    mortgage["year_month"] = mortgage["date"].dt.to_period("M")
    mortgage_monthly = (
        mortgage.groupby("year_month", as_index=False)["rate_30yr_fixed"]
        .mean()
    )
    return mortgage_monthly


def merge_mortgage_rates(frame, mortgage_monthly, date_column, dataset_name):
    frame_with_key = frame.copy()
    frame_with_key["year_month"] = pd.to_datetime(
        frame_with_key[date_column],
        errors="coerce",
    ).dt.to_period("M")

    merged = frame_with_key.merge(mortgage_monthly, on="year_month", how="left")
    null_rate_count = int(merged["rate_30yr_fixed"].isnull().sum())
    print(f"{dataset_name} null rate count after merge: {null_rate_count}")

    if null_rate_count != 0:
        raise ValueError(
            f"{dataset_name} merge produced {null_rate_count} null rate values."
        )

    return merged


def save_enriched_outputs(file_path, sold_with_rates, listings_with_rates):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    sold_output = file_path / "CRMLSSold_filtered_residential_with_rates.csv"
    listings_output = file_path / "CRMLSListing_filtered_residential_with_rates.csv"

    sold_with_rates.to_csv(sold_output, index=False)
    listings_with_rates.to_csv(listings_output, index=False)

    print(f"Enriched sold dataset saved to {sold_output}")
    print(f"Enriched listings dataset saved to {listings_output}")


if __name__ == "__main__":
    mortgage_monthly = fetch_mortgage_monthly_rates()
    sold = process_prefix("CRMLSSold")
    listings = process_prefix("CRMLSListing")

    if sold is None or listings is None:
        raise RuntimeError("Unable to build the sold and listings datasets for merging.")

    sold_with_rates = merge_mortgage_rates(sold, mortgage_monthly, "CloseDate", "Sold")
    listings_with_rates = merge_mortgage_rates(
        listings,
        mortgage_monthly,
        "ListingContractDate",
        "Listings",
    )

    print(
        sold_with_rates[
            ["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]
        ].head()
    )

    save_enriched_outputs(SOURCE_DIR, sold_with_rates, listings_with_rates)

# ===========================
# WEEK 4-5 DATA QUALITY TOOLS
# ===========================

TRANSFORMATION_LOG = [
    ("Concatenated monthly files", "Combined all monthly MLS exports into one dataset."),
    ("Filtered Residential records", "Project scope is residential properties only."),
    ("Converted date columns", "Ensures consistent datetime analysis."),
    ("Merged mortgage rates", "Adds monthly economic context.")
]

def print_transformation_log():
    print("\nTransformation Log")
    for step, reason in TRANSFORMATION_LOG:
        print(f"- {step}: {reason}")

def print_row_count_summary(before_rows, after_rows):
    print("\nRow Count Summary")
    print(f"Original rows : {before_rows}")
    print(f"Cleaned rows  : {after_rows}")
    print(f"Rows removed  : {before_rows-after_rows}")

def save_dtype_summary(frame, out_dir):
    report = pd.DataFrame({
        "Column": frame.columns,
        "DataType": frame.dtypes.astype(str).values
    })
    report.to_csv(Path(out_dir) / "datatype_summary.csv", index=False)

def date_consistency_summary(frame):
    cols = ["ListingContractDate","CloseDate","PurchaseDate"]
    for c in cols:
        if c in frame.columns:
            frame[c] = pd.to_datetime(frame[c], errors="coerce")

    invalid = pd.Series(False, index=frame.index)

    if {"ListingContractDate","CloseDate"}.issubset(frame.columns):
        invalid |= frame["CloseDate"] < frame["ListingContractDate"]

    if {"PurchaseDate","CloseDate"}.issubset(frame.columns):
        invalid |= frame["PurchaseDate"] > frame["CloseDate"]

    print(f"\nDate consistency violations: {int(invalid.sum())}")
    return int(invalid.sum())

def geographic_summary(frame):
    if not {"Latitude","Longitude"}.issubset(frame.columns):
        print("\nLatitude/Longitude columns not found.")
        return 0

    lat = pd.to_numeric(frame["Latitude"], errors="coerce")
    lon = pd.to_numeric(frame["Longitude"], errors="coerce")

    invalid = (
        lat.isna() |
        lon.isna() |
        (lat < -90) | (lat > 90) |
        (lon < -180) | (lon > 180) |
        (lat == 0) | (lon == 0)
    )

    print(f"Invalid coordinate records: {int(invalid.sum())}")
    return int(invalid.sum())

Found 25 monthly files for CRMLSSold from 2024-01 through 2026-06.
Row count before concatenation: 545481
Row count after concatenation: 545481
Filtering logic applied: keep rows where PropertyType equals Residential.
Row count before Residential filter: 545481
Row count after Residential filter: 367741
Unique property types found for CRMLSSold: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']

Null-count summary table:
                      column  null_count  null_percent
               BuyerAgentAOR       75925     13.918908
                ListAgentAOR       68025     12.470645
                    Flooring      225437     41.328112
                      ViewYN       55107     10.102460
                WaterfrontYN      545179     99.944636
                  BasementYN      536504     98.354296
               PoolPrivateYN       65989     12.097397
           OriginalListPrice        16

# WK 6 

In [32]:
# Submit a .py script demonstrating all engineered metrics (price ratio, close-to-original-list ratio, PPSF, days
# on market, YrMo, listing-to-contract days, contract-to-close days), with a sample output table showing the
# new columns populated correctly. Include at least one segmented summary table grouped by
# PropertyType or CountyOrParish.

# Price Ratio ClosePrice / OriginalListPrice Measures negotiation strength
# Price Per Sq Ft ClosePrice / LivingArea Normalizes price across sizes
# Days on Market DaysOnMarket (raw field) Time-to-sell indicator
# Year / Month / YrMo Derived from CloseDate Enables time-series analysis
# Close to Original List
# Ratio
# ClosePrice / OriginalListPrice Captures full price reduction history
# Listing to Contract Days PurchaseContractDate -
# ListingContractDate
# Measures time from listing to accepted
# offer
# Contract to Close Days CloseDate - PurchaseContractDate Escrow and closing period duration
# Add school districts using the properties’ latitude and longitude values
# Add school districts using the properties’ latitude and longitude values
# https://data.ca.gov/dataset/california-school-district-areas-2024-25/resource/7dfaf005-58eb-45db-93b1-7aff091b2172